# Spatial Effects (Cartesian): Screen Position and Looking without Seeing

In this notebook we examine the spatial aspects of our paradigm in Cartesian (x, y) coordinates.
<br><br>
First, we look at the spatial distribution of targets and distractors in our task to verify that they are distributed uniformly across the display and across target categories.<br>
Then, we examine _HITS_ and _MISSES_ across space to see if there are any systematic patterns in their distribution.<br>
Lastly, we look at the spatial distribution of Looking without Seeing (LWS; i.e., $P[\text{miss} | \text{on target}]$) to see if there are any patterns that predict where LWS occurs.

See `spatial_polar.ipynb` for the eccentricity/radial-coordinates counterpart, which asks a narrower version of the same question ("does distance from screen center matter?") and a parametric log-linear robustness check.

### Caveat: `px2deg` is position-dependent

`distance_dva` is computed as `distance_px * px2deg`, where `px2deg` is the angle subtended by one pixel
at the **screen centre**. The true angle a fixation-target pair subtends shrinks with eccentricity by a
factor of `D^2 / (D^2 + e^2)`, so the linear conversion overestimates away from centre: measured over the
1,582 real targets, median 3.0%, p90 6.9%, max 9.7% (no target exceeds 10%).

For every other analysis this is immaterial - it amounts to a few percent wobble in a threshold
(`ON_TARGET_THRESHOLD_DVA = 1.75`) that is itself a chosen convention. **This notebook is the exception**,
because the bias is a smooth centre-to-periphery gradient, i.e. the same functional form as the spatial
effect being estimated, and is correlated with the predictor rather than independent of it.

**If a centre-periphery gradient in LWS probability is reported as a finding**, run one robustness check
first: recompute `distance_dva` as the exact angle between the two eye-to-screen vectors rather than
`distance_px * px2deg`, refit, and confirm the smooth's shape is unchanged. Alternatively add target
eccentricity as a covariate so any artefact is absorbed rather than attributed to LWS.
See `CODE_REVIEW.md` M12 and `spatial_polar.ipynb`, which does exactly this (models eccentricity directly).

In [1]:
import os
from itertools import product

import numpy as np
import pandas as pd

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

import config as cnfg
from analysis.helpers.read_data import load_analysis_data
from analysis.helpers.r_bridge import (
    setup_rpy2, to_r_dataframe, source_r, get_r_object, gam_metrics, predict_gam, placeholder_level, cached_fit,
)
from analysis.helpers.spatial_support import compute_grid_support
from analysis.helpers.plotting.heatmaps import plot_screen_fixed_heatmaps
from analysis.helpers.plotting.gam_overlay import plot_gam_spatial_predictions

pio.renderers.default = "notebook"      # "notebook" or "browser"
setup_rpy2()

## Visualize Spatial Counts
We visualize how many instances of each "event" (target existence, missed target, target visits, fixations) occur across the screen. This gives us a visual representation of the spatial distribution of these "events" across the display, and allows us to check for any systematic biases in their distribution (e.g., more targets appearing on the left side of the screen, or more missed targets in the center).

In [2]:
data, funnel_results = load_analysis_data(funnel_type="lws", event_type="visit")
valid_trials = data.trial_funnel.loc[data.trial_funnel["is_valid_trial"], ["subject", "trial"]]
funnel_results = funnel_results.merge(valid_trials, on=["subject", "trial"], how="inner")
targets = data.targets
idents = data.identifications
fixations = data.fixations
visits = data.visits

### Targets

In [3]:
plot_screen_fixed_heatmaps(targets, "category", title="Target Distribution")

### Missed Targets

In [4]:
missed_targets = (
    idents
    .loc[idents["identification_category"] == "miss", ["subject", "trial", "target"]]
    .merge(targets, on=["subject", "trial", "target"])
)
plot_screen_fixed_heatmaps(missed_targets, "category", title="Missed Target Distribution")

### Fixations

In [5]:
plot_screen_fixed_heatmaps(fixations, "subject", title="Fixation Distribution")

### Visits

In [6]:
plot_screen_fixed_heatmaps(visits, "subject", title="Target-Visit Distribution")

### LWS Probability
The ratio between the number of LWS visits and the total number of visits, within each spatial bin

In [7]:
plot_screen_fixed_heatmaps(
    funnel_results, "subject", title="LWS-Visit Probability", mode="probability"
)

## Statistical Analysis of $P[\text{LWS}]$ across Space
We want to answer the question _"Are the locations/regions on the screen that are more likely to elicit LWS?"_.<br>
More technically, we want to see if $P[\text{miss} | \text{on target}]$ varies across different "bins" of the screen, while controlling for subject - across all trials and within each trial category.
<br><br>
To test the statistical significance, we fit a **Generalized Additive Model (GAM)** - a generalization of GLMMs that can model non-linear relationships between predictors and the outcome variable. In our case, we can use a GAM to model the probability of LWS as a smooth function of the spatial coordinates (x, y) of the visits, while including random effects for subjects to account for individual baseline differences (intercepts). The model is specified as follows:
$$\text{logit}(P[\text{miss} | \text{on target}]) \sim C(\text{trial\_category}) + te(x, y) + (1 | \text{subject}) + (1 | \text{trial})$$
Where:
- $C(\text{trial\_category})$ is a categorical predictor of trial category ("color", "noise", "bw")
- $te(x, y)$ is a tensor product smooth term that models the non-linear interaction between the spatial coordinates (x, y) of the target-visit's center
- $(1 | \text{subject})$ is a random intercept for each subject to account for individual differences in baseline LWS probability.
- $(1 | \text{trial})$ is a random intercept nested within subject (one level per subject x trial), added because the unit of observation is a *visit* and visits nest within trial within subject - without it, the smooth's p-value would be anti-conservative (`CODE_REVIEW.md` M10).

**Null hypothesis:** $H_0$ = no effect of screen location $(x, y)$ on $P[\text{LWS}]$ - the fitted surface is flat (constant), independent of where on the screen the target sits.

**Reading the result:** report the smooth's estimated degrees of freedom (edf) alongside its p-value, not the p-value alone. `te(x, y)` has up to ~$K^2$ basis functions (`K=8` here, so up to 64), so a "significant" result with edf near that ceiling is at least as likely to reflect the model using its flexibility to track noise (overfitting) as a real, generalizable spatial pattern - it is not, by itself, strong evidence for $H_1$. A significant result with much lower edf is a simpler, more credible effect. See `spatial_polar.ipynb` for a model that targets a specific, lower-dimensional hypothesis (distance-from-center / angle) rather than an arbitrary 2-D surface.

**M10 severity check:** below we fit both the flat `(1|subject)` model and the nested `(1|trial:subject)` model and compare them directly - goodness-of-fit (AIC/BIC/deviance explained) and `te(x,y)`'s edf/p-value in each - rather than only reporting the corrected one. We also keep `interaction_model` (does the spatial pattern differ by trial category?) from the earlier M10/M11 fix session as-is - a secondary decomposition of the same `te(x,y)` effect, not re-paired here since a separate flat/nested comparison on it would very likely just re-demonstrate the same M10 effect a third time.

`mgcv` is fit directly in this notebook via `rpy2` (`analysis/helpers/r_bridge.py`) - no separate `Rscript` run, no CSV export/import.

In [8]:
CENTER_X, CENTER_Y = cnfg.TOBII_MONITOR.width / 2, cnfg.TOBII_MONITOR.height / 2


def _fit_spatial_cartesian():
    to_r_dataframe(funnel_results[["subject", "trial", "trial_category", "x", "y", "is_lws"]], "dat")
    source_r(os.path.join(os.getcwd(), "R", "spatial_cartesian_gam.R"))
    model_flat = get_r_object("simple_model_flat")
    model_nested = get_r_object("simple_model_nested")
    model_interaction = get_r_object("interaction_model")

    x_vals = np.linspace(funnel_results["x"].min(), funnel_results["x"].max(), 160)
    y_vals = np.linspace(funnel_results["y"].min(), funnel_results["y"].max(), 90)
    categories = funnel_results["trial_category"].cat.categories.tolist()
    subjects = funnel_results["subject"].astype(str).unique().tolist()
    grid = pd.DataFrame(product(x_vals, y_vals, categories, subjects), columns=["x", "y", "trial_category", "subject"])
    is_supported = compute_grid_support(grid, funnel_results[["x", "y"]])

    preds_flat = grid.assign(prob=predict_gam(model_flat, grid), is_supported=is_supported)
    grid_nested = grid.assign(trial_uid=placeholder_level("dat$trial_uid"))
    preds_nested = grid.assign(
        prob=predict_gam(model_nested, grid_nested, exclude=["s(trial_uid)"]), is_supported=is_supported,
    )

    return {
        "metrics_flat": gam_metrics(model_flat, "flat (1|subject)"),
        "metrics_nested": gam_metrics(model_nested, "nested (1|trial:subject)"),
        "metrics_interaction": gam_metrics(model_interaction, "interaction (nested, as-fit this session)"),
        "preds_flat": preds_flat,
        "preds_nested": preds_nested,
    }


result = cached_fit(os.path.join(os.getcwd(), "R", "_cache", "spatial_cartesian.pkl"), _fit_spatial_cartesian)

for m in (result["metrics_flat"], result["metrics_nested"], result["metrics_interaction"]):
    print(f"--- {m['label']} --- AIC={m['aic']:.1f}  BIC={m['bic']:.1f}  R-sq.adj={m['r_sq_adj']:.4f}  dev.expl={m['dev_expl']:.4f}")
    display(m["smooth_terms"])

plot_gam_spatial_predictions(result["preds_flat"], title="GAM-Predicted LWS Probability Surface - flat (1|subject)").show()
plot_gam_spatial_predictions(result["preds_nested"], title="GAM-Predicted LWS Probability Surface - nested (1|trial:subject)").show()

C:\Users\nirjo\Documents\University\PhD\Projects\LWSv1\.venv\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to convert the column "subject". Fall back to string conversion. The error is: Converting pandas "Category" series to R factor is only possible when categories are strings.
  warnings.warn('Error while trying to convert '
R callback write-console: Loading required package: nlme
  


R callback write-console: This is mgcv 1.9-4. For overview type '?mgcv'.
  


R callback write-console: In addition:   


R callback write-console: Warning messages:
  


R callback write-console: 1: package 'mgcv' was built under R version 4.5.3 
  


...

,term,edf,ref_df,chi_sq,p-value
0,"te(x,y)",7.138858,8.563054,16.169027,0.059133
1,s(subject),22.581287,27.000000,139.518540,0.000000


--- nested (1|trial:subject) --- AIC=6259.3  BIC=7824.2  R-sq.adj=0.0851  dev.expl=0.1048


,term,edf,ref_df,chi_sq,p-value
0,"te(x,y)",6.636218,7.858517,13.508604,0.112876
1,s(subject),21.393395,27.000000,146.836333,0.000000
2,s(trial_uid),201.725384,1361.000000,252.027833,0.000000


--- interaction (nested, as-fit this session) --- AIC=6270.0  BIC=7906.9  R-sq.adj=0.0862  dev.expl=0.1065


,term,edf,ref_df,chi_sq,p-value
0,"te(x,y)",4.091684,5.384973,13.908475,0.025690
1,"te(x,y):trial_categoryCOLOR",7.582193,9.341523,8.716281,0.489123
2,"te(x,y):trial_categoryBW",5.350998,6.309280,7.947516,0.284724
3,"te(x,y):trial_categoryNOISE",3.158221,3.296221,2.860562,0.439808
4,s(subject),21.415737,27.000000,144.760898,0.000000
5,s(trial_uid),193.559496,1361.000000,239.462513,0.000000
